This is the version 2 NB-04.

Cell 1 — Install & Load Kaggle Secret

In [1]:
!pip install -q -U openai

import os
import json
import glob
import time
import concurrent.futures
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, recall_score

SEED = 42

# Load OpenRouter API key from Kaggle Secrets (or fallback to env var)
try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_API_KEY = UserSecretsClient().get_secret("311TuneKey")
except Exception:
    OPENROUTER_API_KEY = os.getenv("311TuneKey", "")

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY not found! Add it in Kaggle Add-ons -> Secrets."

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)
print("OpenRouter client initialized.")

OpenRouter client initialized.


2 — Configuration, Model IDs & SAMPLE_N Toggle
Keep SAMPLE_N = 20 for your first smoke test (costs ~$0.05), then set SAMPLE_N = None to run all 343 validation rows.

In [2]:
# Set to 20 for a quick smoke test; set to None for the full 343-row validation split
SAMPLE_N = 20

# OpenRouter model IDs & approximate pricing ($ per 1M tokens: input, output)
MODELS = {
    "haiku":  {"id": "anthropic/claude-3.5-haiku",  "in_per_m": 0.80, "out_per_m": 4.00},
    "sonnet": {"id": "anthropic/claude-3.5-sonnet", "in_per_m": 3.00, "out_per_m": 15.00},
    "qwen3_4b": {"id": "qwen/qwen3-4b:free", "in_per_m": 0.0, "out_per_m": 0.0},
}

# Locked 10 route codes (Interface 1 & Interface 2)
ROUTE_CODES = [
    "SW-BIN", "SW-MISS", "AN-DEAD", "AN-INJ", "RD-POT",
    "TR-TREE", "PS-PROP", "IL-DUMP", "EMERG", "OOS"
]

PROMPTS_DIR = "/kaggle/working/prompts/classifier"
RESULTS_DIR = "/kaggle/working/results/prompted"
os.makedirs(PROMPTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

3 — Load train.jsonl & Recreate Exact NB-05 Split
Uses the exact same GroupShuffleSplit(test_size=0.15, random_state=42) on template_id so Haiku and Sonnet are evaluated on the exact same 343 validation rows as your DistilRoBERTa and ModernBERT runs.

In [3]:
matches = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert matches, f"train.jsonl not found! Available: {glob.glob('/kaggle/input/**/*', recursive=True)[:15]}"

TRAIN_JSONL_PATH = matches[0]
with open(TRAIN_JSONL_PATH, "r", encoding="utf-8") as f:
    df = pd.DataFrame([json.loads(line) for line in f if line.strip()])

df = df[df["service_code"].isin(ROUTE_CODES)].copy().reset_index(drop=True)

group_col = "template_id" if "template_id" in df.columns else ("group" if "group" in df.columns else None)
groups = df[group_col] if group_col else df.index

splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, df["service_code"], groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

eval_df = val_df.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True) if SAMPLE_N else val_df.copy()
print(f"Train fold: {len(train_df)} | Full Val fold: {len(val_df)} | Running eval on: {len(eval_df)} rows")

Train fold: 1916 | Full Val fold: 343 | Running eval on: 20 rows


4- Build & Save zero_shot.txt and few_shot.txt
Pulls 1 representative example per route code strictly from train_df (never val_df or test.jsonl), writes both files to /kaggle/working/prompts/classifier/, and prints them so you can inspect or paste them into Supabase config_files on /admin.

In [4]:
ZERO_SHOT_PROMPT = """You are a Municipal 311 Intake Classifier.
Classify the resident's message into EXACTLY ONE of the 10 route codes below:

- SW-BIN: Damaged, broken, or missing waste/recycling/organics/yard-waste bin
- SW-MISS: Missed curbside waste, recycling, or organics collection
- AN-DEAD: Dead animal on public roadway, sidewalk, or park
- AN-INJ: Injured, distressed, trapped, or aggressive animal
- RD-POT: Pothole, sinkhole, or road/sidewalk surface damage
- TR-TREE: City tree pruning, fallen limb, or leaning tree
- PS-PROP: Property standards violation, vital services (no heat/water), structural danger, or bylaw upkeep
- IL-DUMP: Illegal dumping of furniture, debris, or hazardous material on public land
- EMERG: Immediate life-safety emergency (active fire, smoke from a building, gas smell/leak, sparking/exposed wires, open manhole, violent crime). Note: a routine "fire hydrant" mention without danger is NOT EMERG.
- OOS: Out-of-scope inquiry (property taxes, parking tickets, noise, general questions) or non-English requests without an immediate life-safety emergency.

Rules:
1. Choose ONLY from the 10 codes above.
2. Provide a calibrated confidence score between 0.0 and 1.0.
3. Provide the top 2 most likely codes in descending order of confidence.
4. Return ONLY valid JSON matching this exact schema (no markdown, no explanation):
{"code": "SW-BIN", "confidence": 0.92, "top2": [{"code": "SW-BIN", "confidence": 0.92}, {"code": "SW-MISS", "confidence": 0.05}]}
"""

# Sample 1 clean example per code strictly from the 85% train_df fold
few_shot_lines = []
for code in ROUTE_CODES:
    subset = train_df[train_df["service_code"] == code]
    # Prefer English, easy/complete examples where available for clear anchors
    if "lang" in subset.columns and (subset["lang"] == "en").any():
        subset = subset[subset["lang"] == "en"]
    sample_row = subset.sample(n=1, random_state=SEED).iloc[0]
    msg = sample_row["message"].replace("\n", " ").strip()
    runner_up = "OOS" if code != "OOS" else "SW-BIN"
    example_json = json.dumps({
        "code": code,
        "confidence": 0.95,
        "top2": [{"code": code, "confidence": 0.95}, {"code": runner_up, "confidence": 0.03}],
    })
    few_shot_lines.append(f'Message: "{msg}"\nOutput: {example_json}')

FEW_SHOT_PROMPT = ZERO_SHOT_PROMPT + "\nExamples:\n" + "\n\n".join(few_shot_lines) + "\n"

# Save both prompt files for Supabase config_files
with open(f"{PROMPTS_DIR}/zero_shot.txt", "w", encoding="utf-8") as f:
    f.write(ZERO_SHOT_PROMPT)
with open(f"{PROMPTS_DIR}/few_shot.txt", "w", encoding="utf-8") as f:
    f.write(FEW_SHOT_PROMPT)

print(f"Saved zero_shot.txt ({len(ZERO_SHOT_PROMPT)} chars) and few_shot.txt ({len(FEW_SHOT_PROMPT)} chars)")

Saved zero_shot.txt (1473 chars) and few_shot.txt (5632 chars)


Cell 5 — Interface 4 Prompted Classifier Function
Implements the exact Interface 4 return dictionary (code, confidence, top2, setup, model, prompt_version, tokens_in, tokens_out, cost_usd, latency_ms) with strict 10-code validation and OOS fallback.

In [5]:
SETUPS = {
    "haiku_zeroshot":  {"model_key": "haiku",  "prompt": ZERO_SHOT_PROMPT, "prompt_version": "clf-zs-v1"},
    "haiku_fewshot":   {"model_key": "haiku",  "prompt": FEW_SHOT_PROMPT,  "prompt_version": "clf-fs-v1"},
    "sonnet_zeroshot": {"model_key": "sonnet", "prompt": ZERO_SHOT_PROMPT, "prompt_version": "clf-zs-v1"},
    "sonnet_fewshot":  {"model_key": "sonnet", "prompt": FEW_SHOT_PROMPT,  "prompt_version": "clf-fs-v1"},
    "qwen3_4b_fewshot": {"model_key": "qwen3_4b", "prompt": FEW_SHOT_PROMPT, "prompt_version": "clf-fs-v1"},
}

def classify_prompted(text: str, setup_name: str) -> dict:
    cfg = SETUPS[setup_name]
    m_info = MODELS[cfg["model_key"]]
    model_id = m_info["id"]

    t0 = time.perf_counter()
    tokens_in, tokens_out, cost_usd = 0, 0, 0.0
    raw = ""

    try:
        resp = client.chat.completions.create(
            model=model_id,
            temperature=0.0,
            max_tokens=120,
            messages=[
                {"role": "system", "content": cfg["prompt"]},
                {"role": "user", "content": text},
            ],
        )
        raw = (resp.choices[0].message.content or "").strip()
        if resp.usage:
            tokens_in = resp.usage.prompt_tokens
            tokens_out = resp.usage.completion_tokens
            cost_usd = (tokens_in * m_info["in_per_m"] + tokens_out * m_info["out_per_m"]) / 1_000_000
    except Exception:
        pass

    latency_ms = int((time.perf_counter() - t0) * 1000)

    # Parse & validate JSON against the 10 route codes (OOS fallback on parse failure)
    code, conf, top2 = "OOS", 0.0, []
    try:
        clean = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        start, end = clean.find("{"), clean.rfind("}")
        parsed = json.loads(clean[start:end + 1]) if start != -1 and end != -1 else {}

        c = str(parsed.get("code", "")).strip().upper()
        code = c if c in ROUTE_CODES else "OOS"
        conf = float(parsed.get("confidence", 0.0)) if c in ROUTE_CODES else 0.0

        for item in (parsed.get("top2") or [])[:2]:
            tc = str(item.get("code", "OOS")).strip().upper()
            tp = float(item.get("confidence", 0.0))
            top2.append({"code": tc if tc in ROUTE_CODES else "OOS", "confidence": round(tp, 4)})
    except Exception:
        pass

    if len(top2) < 2:
        second = "OOS" if code != "OOS" else "SW-BIN"
        top2 = [{"code": code, "confidence": round(conf, 4)}, {"code": second, "confidence": round(max(0.0, 1.0 - conf), 4)}]

    return {
        "code": code,
        "confidence": round(conf, 4),
        "top2": top2,
        "setup": setup_name,
        "model": model_id,
        "prompt_version": cfg["prompt_version"],
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
        "cost_usd": round(cost_usd, 6),
        "latency_ms": latency_ms,
    }

Cell 6 — Run All 4 Prompted Setups (Parallelized)
Uses 6 concurrent threads so 343 rows finish in ~1–2 minutes per setup instead of 15+ minutes.

In [6]:
all_records = []

for setup_name in SETUPS.keys():
    print(f"\nRunning {setup_name} on {len(eval_df)} rows...")
    t_start = time.perf_counter()

    def _run_row(row_tuple):
        idx, row = row_tuple
        out = classify_prompted(row["message"], setup_name)
        top2_margin = round(out["top2"][0]["confidence"] - out["top2"][1]["confidence"], 4)
        return {
            "setup": setup_name,
            "id": row.get("id", idx),
            "true_code": row["service_code"],
            "pred_code": out["code"],
            "confidence": out["confidence"],
            "top2_margin": top2_margin,
            "lang": row.get("lang", "en"),
            "difficulty": row.get("difficulty", "unknown"),
            "tokens_in": out["tokens_in"],
            "tokens_out": out["tokens_out"],
            "cost_usd": out["cost_usd"],
            "latency_ms": out["latency_ms"],
        }

    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as ex:
        setup_rows = list(ex.map(_run_row, eval_df.iterrows()))

    all_records.extend(setup_rows)
    elapsed = time.perf_counter() - t_start
    acc = np.mean([r["true_code"] == r["pred_code"] for r in setup_rows])
    tot_cost = sum(r["cost_usd"] for r in setup_rows)
    print(f"Done {setup_name} in {elapsed:.1f}s | Acc: {acc:.4f} | Total Cost: ${tot_cost:.4f}")

preds_df = pd.DataFrame(all_records)
preds_df.to_csv(f"{RESULTS_DIR}/nb04_prompted_predictions.csv", index=False)


Running haiku_zeroshot on 20 rows...
Done haiku_zeroshot in 0.3s | Acc: 0.0500 | Total Cost: $0.0000

Running haiku_fewshot on 20 rows...
Done haiku_fewshot in 2.1s | Acc: 0.0500 | Total Cost: $0.0000

Running sonnet_zeroshot on 20 rows...
Done sonnet_zeroshot in 0.2s | Acc: 0.0500 | Total Cost: $0.0000

Running sonnet_fewshot on 20 rows...
Done sonnet_fewshot in 0.1s | Acc: 0.0500 | Total Cost: $0.0000

Running qwen3_4b_fewshot on 20 rows...
Done qwen3_4b_fewshot in 0.5s | Acc: 0.0500 | Total Cost: $0.0000


7 — Summary Comparison Table (Prompted vs. Your finetuned_v1 Baselines)
Computes Accuracy, Macro-F1, EMERG Recall, OOS Recall, Rule E3 trigger rate, Latency (p50/p95), and Cost per 1,000 calls.

In [7]:
summary_rows = []

for setup_name, g in preds_df.groupby("setup", sort=False):
    acc = accuracy_score(g["true_code"], g["pred_code"])
    macro_f1 = f1_score(g["true_code"], g["pred_code"], labels=ROUTE_CODES, average="macro", zero_division=0)

    emerg_g = g[g["true_code"] == "EMERG"]
    emerg_rec = (emerg_g["true_code"] == emerg_g["pred_code"]).mean() if len(emerg_g) else np.nan

    oos_g = g[g["true_code"] == "OOS"]
    oos_rec = (oos_g["true_code"] == oos_g["pred_code"]).mean() if len(oos_g) else np.nan

    e3_rate = ((g["confidence"] < 0.70) | (g["top2_margin"] < 0.10)).mean()

    summary_rows.append({
        "Setup": setup_name,
        "N": len(g),
        "Accuracy": round(acc, 4),
        "Macro-F1": round(macro_f1, 4),
        "EMERG Recall": round(float(emerg_rec), 4) if not np.isnan(emerg_rec) else None,
        "OOS Recall": round(float(oos_rec), 4) if not np.isnan(oos_rec) else None,
        "E3 Trigger Rate": round(float(e3_rate), 4),
        "P50 Latency (ms)": int(np.median(g["latency_ms"])),
        "P95 Latency (ms)": int(np.percentile(g["latency_ms"], 95)),
        "Cost / 1k Calls ($)": round(float(g["cost_usd"].mean() * 1000), 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{RESULTS_DIR}/nb04_prompted_summary.csv", index=False)
print(summary_df.to_string(index=False))

           Setup  N  Accuracy  Macro-F1 EMERG Recall  OOS Recall  E3 Trigger Rate  P50 Latency (ms)  P95 Latency (ms)  Cost / 1k Calls ($)
  haiku_zeroshot 20      0.05    0.0095         None         1.0              1.0                34               207                  0.0
   haiku_fewshot 20      0.05    0.0095         None         1.0              1.0                30               869                  0.0
 sonnet_zeroshot 20      0.05    0.0095         None         1.0              1.0                30                51                  0.0
  sonnet_fewshot 20      0.05    0.0095         None         1.0              1.0                25                39                  0.0
qwen3_4b_fewshot 20      0.05    0.0095         None         1.0              1.0                26                65                  0.0


8 — Zip & Generate Download Links for Prompts + Results

In [8]:
from IPython.display import FileLink, display

!cd /kaggle/working && zip -rq nb04_prompts_and_results.zip prompts/classifier results/prompted
os.chdir("/kaggle/working")
display(FileLink("nb04_prompts_and_results.zip"))

/kaggle/working/nb04_prompts_and_results.zip